# 32 整数规划与目标规划

> **模块**：优化 ｜ **数据**：无（内置示例） ｜ **依赖**：numpy、pulp

## 1. 问题背景

整数规划要求部分或全部变量取整，目标规划则用偏差变量处理多目标冲突。两者都是线性规划的扩展，常见于排班、选址与多目标决策。

## 2. 学习目标

- 用 `pulp` 建立整数规划模型并求解
- 理解目标规划中正、负偏差变量的含义
- 能解读求解状态与灵敏度信息

## 3. 环境与数据

- 依赖：numpy、pulp
- 数据：无（内置示例）

## 4. 方法与实现

### 4.1 整数规划

In [1]:
# Integer Program Problem
'''
here is an example to solve the following problem:
max z = x[0]**2 + x[1]**2 + 3*x[2]**2 + 4*x[3]**2 + 2*x[4]**2 - 8*x[0] - 2*x[1] - 3*x[2] - x[3] - 2*x[4]
s.t   x in bounds
      A_ub * x <= b_ub
here x is an array of integers.
'''

import numpy as np

A_ub = [[1, 1, 1, 1, 1],
        [1, 2, 2, 1, 6],
        [2, 1, 6, 0, 0],
        [0, 0, 1, 1, 5]]

b_ub = [400, 800, 200, 200]
bounds = [(0, 99), (0, 99), (0, 99), (0, 99)]

def z(x):
    return x[0]**2 + x[1]**2 + 3*x[2]**2 + 4*x[3]**2 + 2*x[4]**2 - 8*x[0] - 2*x[1] - 3*x[2] - x[3] - 2*x[4]

In [2]:
# Monte Carlo Method 蒙特卡洛方法

x0 = []
z0 = 0
for i in range(int(1000000)):
    x = np.random.randint(low=0, high=99, size=5, dtype=np.int64)
    if(np.max(np.matmul(A_ub, x)-b_ub) > 0):
        continue
    tempz = z(x)
    if tempz > z0:
        z0 = tempz
        x0 = x
        
print('The best z is :', z0)
print('The according x is: ', x0)

The best z is : 47945
The according x is:  [46 94  2 97  2]


In [ ]:
from pulp import *

# 1. 建立问题
prob = LpProblem("Bleding Problem", LpMinimize)

# 2. 建立变量
x1 = LpVariable("ChickenPercent", 0, None, LpInteger)
x2 = LpVariable("BeefPercent", 0)

# 3. 设置目标函数
prob += 0.013*x1 + 0.008*x2, "Total Cost of Ingredients per can"

# 4. 施加约束
prob += x1 + x2 == 100, "PercentagesSum"
prob += 0.100*x1 + 0.200*x2 >= 8.0, "ProteinRequirement"
prob += 0.080*x1 + 0.100*x2 >= 6.0, "FatRequirement"
prob += 0.001*x1 + 0.005*x2 <= 2.0, "FibreRequirement"
prob += 0.002*x2 + 0.005*x2 <= 0.4, "SaltRequirement"

# 5. 求解
prob.solve()

# 6. 打印求解状态
print("Status:", LpStatus[prob.status])

# 7. 打印出每个变量的最优值
for v in prob.variables():
    print(v.name, "=", v.varValue)

# 8. 打印最优解的目标函数值
print("Total Cost of Ingredients per can = ", value(prob.objective))

In [ ]:
# 例2.8

from pulp import *

# 1. 建立问题
prob = LpProblem("MIP", LpMinimize)

# 2. 建立变量
x = LpVariable.dicts(name="x", indexs=[1, 2], lowBound=0, cat='Continuous')
x3 = LpVariable(name='x3', lowBound=0, upBound=1, cat='Integer')
x[2].setInitialValue(5.5)
x3.setInitialValue(1)

# 3. 设置目标函数
prob += -3*x[1]-2*x[2]-x3

# 4. 施加约束
prob += x[1] + x[2] + x3 <= 7
prob += 4*x[1] + 2*x[2] + x3 == 12

# 5. 求解
prob.solve()

# 6. 打印求解状态
print("Status:", LpStatus[prob.status])

# 7. 打印出每个变量的最优值
for v in prob.variables():
    print(v.name, "=", v.varValue)

# 8. 打印最优解的目标函数值
print("Total Cost of Ingredients per can = ", value(prob.objective))

### 4.2 目标规划

**基本上分为两种思路: 加权系数法(转化为单一目标), 优先等级法(按重要程度不同, 转化为单目标模型)**

正负偏差变量: $d_i^+=max\{f_i-d_i^0,0\},\ d_i^-=-min\{f_i-d_i^0,0\}$, 显然恒有$d_i^+\times d_i^-=0$

In [ ]:
import pulp
prob = pulp.LpProblem("GoalProgramming", pulp.LpMinimize)

# 决策变量
x      = [pulp.LpVariable(f"x{i}",      lowBound=0) for i in range(2)]
dminus = [pulp.LpVariable(f"dminus{i}", lowBound=0) for i in range(4)]
dplus  = [pulp.LpVariable(f"dplus{i}",  lowBound=0) for i in range(4)]

# 硬约束
prob += 2*x[0] + 2*x[1] <= 12

# 目标约束（软约束）
prob += 200*x[0] + 300*x[1] + dminus[0] - dplus[0] == 1500
prob += 2*x[0]   - x[1]     + dminus[1] - dplus[1] == 0
prob += 4*x[0]              + dminus[2] - dplus[2] == 16
prob +=              5*x[1] + dminus[3] - dplus[3] == 15

# ---------- 按优先级逐层求解（字典序） ----------
# 注意：约束 d- 与 d+ 不能同时非零，LP 会自动满足这个性质
priority_terms = [
    dminus[0],                                  # P[0]
    dplus[1] + dminus[1],                       # P[1]
    3*(dplus[2] + dminus[2]) + dplus[3],        # P[2]
]

EPS = 1e-7
for k, term in enumerate(priority_terms):
    prob.setObjective(term)
    status = prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[status] != "Optimal":
        raise RuntimeError(f"第 {k} 优先级不可行 (status={pulp.LpStatus[status]})")
    zk = pulp.value(term)
    print(f"P[{k}] 最优偏差 = {zk:.6f}")
    # 把本层最优值固定为约束，再求下一层
    prob += term <= zk + EPS

# ---------- 结果 ----------
print("\n===== 最终解 =====")
print("x      =", [round(pulp.value(v), 4) for v in x])
print("dminus =", [round(pulp.value(v), 4) for v in dminus])
print("dplus  =", [round(pulp.value(v), 4) for v in dplus])

按目标规划中 \(P[0]\gg P[1]\gg P[2]\) 的优先级求解，最优解为：

$$\boxed{x[0]=2,\quad x[1]=4}$$

各偏差变量为：

$$\begin{aligned}&d_{0}^{-}=0,\quad d_{0}^{+}=100,\\&d_{1}^{-}=0,\quad d_{1}^{+}=0,\\&d_{2}^{-}=8,\quad d_{2}^{+}=0,\\&d_{3}^{-}=0,\quad d_{3}^{+}=5.\end{aligned}$$

代入约束验证：

$$\begin{gathered}2x_{0}+2x_{1}=2(2)+2(4)=12\leq12\\200x_{0}+300x_{1}+d_{0}^{-}-d_{0}^{+}=400+1200+0-100=1500\\2x_{0}-x_{1}+d_{1}^{-}-d_{1}^{+}=4-4+0-0=0\\4x_{0}+d_{2}^{-}-d_{2}^{+}=8+8-0=16\\\\5x_{1}+d_{3}^{-}-d_{3}^{+}=20+0-5=15\end{gathered}$$

目标函数值为：

$$\begin{aligned}z_{\mathrm{min}}&=P[0]d_0^-+P[1](d_1^++d_1^-)+P[2]\left(3(d_2^++d_2^-)+d_3^+\right)\\&=P[0]\cdot0+P[1]\cdot0+P[2]\left(3(0+8)+5\right)\\&=29P[2].\end{aligned}$$

因此最优解为：

$$\begin{gathered}\boxed{x[0]=2,\quad x[1]=4}\\\boxed{d_{0}^{-}=0,\:d_{0}^{+}=100,\quad d_{1}^{-}=0,\:d_{1}^{+}=0,\quad d_{2}^{-}=8,\:d_{2}^{+}=0,\quad d_{3}^{+}=5}\\\boxed{z_{\min}=29P[2]}\end{gathered}$$

$$\begin{aligned}&\text{其中第一优先级达成值 }d_{0}^{-}=0\text{，第二优先级达成值为 29。若}\\&P[0]，P[1],P[2]\text{ 是普通数值权重而非优先级符号，则需要给出具体权重后才能确定数值最优解。}\end{aligned}$$


## 5. 小结

`pulp` 建模按“变量—目标—约束—求解”四步进行；目标规划把多个目标写成偏差之和，再最小化总偏差。

> 内容来源：教材参考实现《数学建模算法与应用》练习稿（MIT，Zhanghan Wang），原始文件已并入本篇，见 `THIRD-PARTY.md`。